In [1]:
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
import torch

In [2]:
domains = {'Politics': ('glue', 'stsb', 'validation[:300]'),'Sports': ('sick', None, 'train[:300]'),'Medicine': ('biosses', None, 'train[:100]')}
column_map = {'glue': ('sentence1', 'sentence2', 'label'),'sick': ('sentence_A', 'sentence_B', 'relatedness_score'),'biosses': ('sentence1', 'sentence2', 'score')}

def evaluate_domain(dataset_name, subset, split):
    if subset:
        dataset=load_dataset(dataset_name, subset, split=split)
    else:
        dataset=load_dataset(dataset_name, split=split)

    s1_col,s2_col,score_col=column_map[dataset_name]

    sent1=dataset[s1_col]
    sent2=dataset[s2_col]
    true_scores=np.array(dataset[score_col])

In [3]:
def hf_encode(model_name, sentences):
    tokenizer=AutoTokenizer.from_pretrained(model_name)
    model=AutoModel.from_pretrained(model_name)

    embeddings=[]
    batch_size=16

    for i in range(0, len(sentences),batch_size):
        batch=sentences[i:i+batch_size]
        inputs=tokenizer(batch,padding=True,truncation=True,return_tensors="pt")

        with torch.no_grad():
            outputs=model(**inputs)

        emb=outputs.last_hidden_state.mean(dim=1)
        embeddings.append(emb)

    return torch.cat(embeddings),model

In [4]:
model=['sentence-transformers/all-MiniLM-L6-v2','BAAI/bge-base-en-v1.5','sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2','nomic-ai/nomic-embed-text-v1.5','jinaai/jina-embeddings-v5-text-nano-retrieval']

In [5]:
res=[]

dataset_name,subset,split_str=domains['Politics']

if subset:
    dataset=load_dataset(dataset_name, subset, split=split_str)
else:
    dataset=load_dataset(dataset_name, split=split_str)

s1_col,s2_col,score_col=column_map[dataset_name]

sent1=dataset[s1_col]
sent2=dataset[s2_col]
true_scores=np.array(dataset[score_col])

for m in model:
    print('Model',m,'\n')
    s=time.time()

    if "sentence-transformers" in m:
      mod=SentenceTransformer(m)
      e1=mod.encode(sent1,convert_to_tensor=True)
      e2=mod.encode(sent2,convert_to_tensor=True)
      size=sum(p.numel() for p in mod._first_module().auto_model.parameters())/1e6
    else:
        e1,model_obj=hf_encode(m,sent1)
        e2, _=hf_encode(m,sent2)
        size=sum(p.numel() for p in model_obj.parameters())/1e6

    sim = cosine_similarity(e1.cpu().float().numpy(),e2.cpu().float().numpy()).diagonal()
    predict=sim*5
    e=time.time()

    pearson=pearsonr(predict,true_scores)[0]
    spearman=spearmanr(predict,true_scores)[0]
    mse=mean_squared_error(true_scores,predict)

    res.append([m,pearson,spearman,mse,e-s,size])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model sentence-transformers/all-MiniLM-L6-v2 



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model BAAI/bge-base-en-v1.5 



...

The repository jinaai/jina-embeddings-v5-text-nano-retrieval contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/jinaai/jina-embeddings-v5-text-nano-retrieval .
 You can inspect the repository content at https://hf.co/jinaai/jina-embeddings-v5-text-nano-retrieval.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


The repository jinaai/jina-embeddings-v5-text-nano-retrieval contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/jinaai/jina-embeddings-v5-text-nano-retrieval .
 You can inspect the repository content at https://hf.co/jinaai/jina-embeddings-v5-text-nano-retrieval.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

In [6]:
df=pd.DataFrame(res, columns=['Model','Pearson','Spearman','MSE','Inference Time','Model Size'])
df

,Model,Pearson,Spearman,MSE,Inference Time,Model Size
0,sentence-transformers/all-MiniLM-L6-v2,0.927166,0.933362,0.513195,10.881097,22.713216
1,BAAI/bge-base-en-v1.5,0.920599,0.934763,2.557526,30.924431,109.482240
2,sentence-transformers/paraphrase-multilingual-...,0.917802,0.923775,0.669671,10.589776,117.653760
3,nomic-ai/nomic-embed-text-v1.5,0.910345,0.924465,3.215248,73.381788,136.731648
4,jinaai/jina-embeddings-v5-text-nano-retrieval,0.815027,0.822158,1.597919,241.765886,211.766016


In [7]:
data=df.iloc[:,1:].values.astype(float)
norm=data/np.sqrt((data**2).sum(axis=0))
weights=np.ones(norm.shape[1])/norm.shape[1]
weighted=norm*weights

ideal_best=np.array([weighted[:,0].max(),weighted[:,1].max(),weighted[:,2].min(),weighted[:,3].min(),weighted[:,4].min()])
worst=np.array([weighted[:,0].min(),weighted[:,1].min(),weighted[:,2].max(),weighted[:,3].max(),weighted[:,4].max()])

dist_best = np.sqrt(((weighted - ideal_best)**2).sum(axis=1))
dist_worst=np.sqrt(((weighted-worst)**2).sum(axis=1))
score=dist_worst/(dist_best+dist_worst)

df["TOPSIS Score"]=score
df["Rank"]=df["TOPSIS Score"].rank(ascending=False)
df

,Model,Pearson,Spearman,MSE,Inference Time,Model Size,TOPSIS Score,Rank
0,sentence-transformers/all-MiniLM-L6-v2,0.927166,0.933362,0.513195,10.881097,22.713216,0.998942,1.0
1,BAAI/bge-base-en-v1.5,0.920599,0.934763,2.557526,30.924431,109.482240,0.625099,3.0
2,sentence-transformers/paraphrase-multilingual-...,0.917802,0.923775,0.669671,10.589776,117.653760,0.778006,2.0
3,nomic-ai/nomic-embed-text-v1.5,0.910345,0.924465,3.215248,73.381788,136.731648,0.484972,4.0
4,jinaai/jina-embeddings-v5-text-nano-retrieval,0.815027,0.822158,1.597919,241.765886,211.766016,0.241272,5.0
